# WFLW_Train — 얼굴 98점 랜드마크 모델 직접 학습 (방법 A)

**만드는 것 (보드로 복사할 파일)**
1. `wflw_mesh.tflite` — 160×160 얼굴 사진 → 98점 좌표(196개 숫자) 회귀 모델 (MobileNetV2 백본)
2. `face_detection_yunet_2023mar.onnx` — OpenCV 얼굴 검출기 (내려받기만 함)
3. `wflw_train_log.png`, `wflw_samples.png` — 발표용 학습 곡선·예측 예시

**미리 준비할 것**
- WFLW 공식 페이지(https://wywu.github.io/projects/LAB/WFLW.html)에서 `WFLW_images.tar.gz`, `WFLW_annotations.tar.gz` 를 받아
  드라이브 `ai_semiconductor/Python/examples/files/WFLW/` (tar.gz 그대로, 압축 변환 불필요) 에 올려 두기 (압축 그대로)
- 이 노트북과 같은 폴더(`proj_face/FaceReading_3/`)를 드라이브에 올려 두기 — `wflw_prep.py` 를 여기서 읽음
- 런타임: **GPU(T4)**. 전처리 5분 + 학습 40분 정도.


## 1. 드라이브 마운트 · 경로

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    colab = True
except Exception:
    colab = False
print('colab' if colab else 'local')

In [ ]:
import os, sys
if colab:
    BASE     = '/content/drive/MyDrive/ai_semiconductor/Python/examples/'
    CODE_DIR = BASE + 'proj_face/FaceReading_3/'          # wflw_prep.py 가 있는 곳 (드라이브에 올린 폴더)
    DATA_DIR = BASE + 'files/WFLW/'                   # tar.gz 두 개
    save_dir = BASE + 'files/save/'
    work_dir = '/content/wflw'
else:
    CODE_DIR = './'; DATA_DIR = '../files/WFLW/'; save_dir = '../files/save/'; work_dir = './wflw'
os.makedirs(save_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)
sys.path.insert(0, CODE_DIR)
from wflw_prep import IMG, FLIP_PERM, crop_box, crop_face, to_input
print('IMG =', IMG)

## 2. 압축 해제 · 라벨 읽기

In [ ]:
!tar -xzf "{DATA_DIR}WFLW_images.tar.gz" -C {work_dir}
!tar -xzf "{DATA_DIR}WFLW_annotations.tar.gz" -C {work_dir}
import glob
IMG_ROOT = glob.glob(work_dir + '/**/WFLW_images', recursive=True)[0] + '/'
ANN_DIR  = glob.glob(work_dir + '/**/list_98pt_rect_attr_train_test', recursive=True)[0] + '/'
print(IMG_ROOT, ANN_DIR)
print(open(ANN_DIR + 'list_98pt_rect_attr_train.txt').readline().split()[-1], '← 첫 줄의 이미지 경로')

In [ ]:
import numpy as np, cv2, time

def load_wflw(txt, root):
    """라벨 txt → (N,IMG,IMG,3) uint8 얼굴 크롭, (N,196) 0~1 좌표"""
    X, Y = [], []
    t0 = time.time()
    for k, line in enumerate(open(txt)):
        t = line.split()
        if len(t) != 207: continue                      # 196 좌표 + 4 박스 + 6 속성 + 1 경로
        pts = np.array(t[:196], np.float32).reshape(98, 2)
        x1, y1, x2, y2 = map(float, t[196:200])
        img = cv2.imread(root + t[-1])
        if img is None: continue
        x0, y0, s = crop_box(x1, y1, x2, y2)             # 보드와 같은 크롭 규칙
        X.append(crop_face(img, x0, y0, s))
        Y.append(((pts - [x0, y0]) / s).reshape(-1))
        if k % 1000 == 0: print(k, '%.0fs' % (time.time() - t0))
    return np.array(X, np.uint8), np.array(Y, np.float32)

cache = work_dir + '/wflw_%d.npz' % IMG
if os.path.exists(cache):
    d = np.load(cache); Xtr, Ytr, Xte, Yte = d['Xtr'], d['Ytr'], d['Xte'], d['Yte']
else:
    Xtr, Ytr = load_wflw(ANN_DIR + 'list_98pt_rect_attr_train.txt', IMG_ROOT)
    Xte, Yte = load_wflw(ANN_DIR + 'list_98pt_rect_attr_test.txt',  IMG_ROOT)
    np.savez_compressed(cache, Xtr=Xtr, Ytr=Ytr, Xte=Xte, Yte=Yte)
print(Xtr.shape, Ytr.shape, Xte.shape, Yte.shape)   # (7500,160,160,3) (7500,196) (2500,...)

점이 얼굴 위에 제대로 찍히는지 눈으로 확인합니다. 점이 얼굴을 벗어나면 크롭 계산이 틀린 것입니다.

In [ ]:
import matplotlib.pyplot as plt
def show(imgs, ptss, cols=6, title=None, pred=None):
    plt.figure(figsize=(2.2 * cols, 2.4))
    for i in range(cols):
        im = imgs[i].copy()
        for x, y in ptss[i].reshape(98, 2) * IMG: cv2.circle(im, (int(x), int(y)), 1, (0, 255, 0), -1)
        if pred is not None:
            for x, y in pred[i].reshape(98, 2) * IMG: cv2.circle(im, (int(x), int(y)), 1, (0, 0, 255), -1)
        plt.subplot(1, cols, i + 1); plt.imshow(im[:, :, ::-1]); plt.axis('off')
    if title: plt.suptitle(title)
    plt.show()
show(Xtr, Ytr, title='train samples (green = label)')

## 3. 증강 (좌우 반전은 점 번호까지 바꿈)

In [ ]:
def augment(img, pts):
    """img (IMG,IMG,3) uint8, pts (98,2) 0~1 → 증강된 쌍"""
    pts = pts.copy()
    if np.random.rand() < 0.5:                                        # 좌우 반전 + 점 번호 교체
        img = img[:, ::-1]; pts[:, 0] = 1 - pts[:, 0]; pts = pts[FLIP_PERM]
    ang = np.random.uniform(-30, 30); sc = np.random.uniform(0.85, 1.15)
    dx, dy = np.random.uniform(-0.06, 0.06, 2) * IMG
    M = cv2.getRotationMatrix2D((IMG / 2, IMG / 2), ang, sc); M[:, 2] += (dx, dy)
    img = cv2.warpAffine(img, M, (IMG, IMG), borderMode=cv2.BORDER_REPLICATE)
    pts = (np.c_[pts * IMG, np.ones(98)] @ M.T) / IMG                 # 좌표에 같은 변환
    img = np.clip(img.astype(np.float32) * np.random.uniform(0.7, 1.3) + np.random.uniform(-25, 25), 0, 255).astype(np.uint8)
    return img, pts

def gen(X, Y, bs=64):
    n = len(X)
    while True:
        idx = np.random.permutation(n)
        for i in range(0, n - bs + 1, bs):
            xb, yb = [], []
            for j in idx[i:i + bs]:
                im, p = augment(X[j], Y[j].reshape(98, 2)); xb.append(im); yb.append(p.reshape(-1))
            xb = np.array(xb, np.float32)[:, :, :, ::-1] / 255.          # BGR→RGB, 0~1  (wflw_prep.to_input 과 동일)
            yield xb, np.array(yb, np.float32)

xb, yb = next(gen(Xtr, Ytr, 6))
show((xb[:, :, :, ::-1] * 255).astype(np.uint8), yb, title='augmented (반전된 샘플도 초록 점이 눈·입에 맞아야 함)')

## 4. 모델 · 학습 (MobileNetV2 → 196개 좌표 회귀, Wing loss)

In [ ]:
import tensorflow as tf
ALPHA = 0.5                                   # 라즈베리파이용. 정확도를 더 원하면 1.0 (속도 2배 느림)
base = tf.keras.applications.MobileNetV2((IMG, IMG, 3), include_top=False, weights='imagenet', alpha=ALPHA)
model = tf.keras.Sequential([
    base, tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'), tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(98 * 2, activation='sigmoid')])         # 0~1 좌표

def wing_loss(y, p, w=0.1, eps=0.02):
    d = tf.abs(y - p); C = w - w * tf.math.log(1 + w / eps)
    return tf.reduce_mean(tf.where(d < w, w * tf.math.log(1 + d / eps), d - C))

model.compile(tf.keras.optimizers.Adam(1e-3), loss=wing_loss)
model.summary()

In [ ]:
%%time
EPOCHS = 60
Xte_in = Xte[:, :, :, ::-1].astype(np.float32) / 255.
hist = model.fit(gen(Xtr, Ytr), steps_per_epoch=len(Xtr) // 64, epochs=EPOCHS,
                 validation_data=(Xte_in, Yte),
                 callbacks=[tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
                            tf.keras.callbacks.ModelCheckpoint(work_dir + '/wflw_best.keras', save_best_only=True)])

## 5. 평가 (NME) · 그림 저장

In [ ]:
model = tf.keras.models.load_model(work_dir + '/wflw_best.keras', custom_objects={'wing_loss': wing_loss})
Pr = model.predict(Xte_in, verbose=0).reshape(-1, 98, 2); G = Yte.reshape(-1, 98, 2)
inter_ocular = np.linalg.norm(G[:, 60] - G[:, 72], axis=1)
nme = (np.linalg.norm(Pr - G, axis=2).mean(1) / inter_ocular).mean() * 100
print('test NME = %.2f %%   (6%% 이하면 과제용으로 충분)' % nme)

plt.figure(figsize=(6, 4)); plt.plot(hist.history['loss'], label='train'); plt.plot(hist.history['val_loss'], label='val')
plt.title('WFLW landmark regression  (test NME %.2f%%)' % nme); plt.xlabel('epoch'); plt.ylabel('wing loss'); plt.grid(True); plt.legend()
plt.savefig(save_dir + 'wflw_train_log.png', dpi=150, bbox_inches='tight'); plt.show()

idx = np.random.choice(len(Xte), 6, replace=False)
show(Xte[idx], Yte[idx], pred=Pr[idx].reshape(6, -1), title='test: green = label, red = our model')
plt.savefig(save_dir + 'wflw_samples.png', dpi=150, bbox_inches='tight')

## 6. TFLite 변환 · 검증 · 저장

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]              # 가중치 양자화 (파일 1/4, 보드 속도 ↑)
tfl = conv.convert()
open(save_dir + 'wflw_mesh.tflite', 'wb').write(tfl)
print('wflw_mesh.tflite  %.1f KB' % (len(tfl) / 1024))

# tflite 결과가 keras 와 같은지 (양자화로 아주 조금 다름)
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i, o = it.get_input_details()[0], it.get_output_details()[0]
errs = []
for k in range(50):
    it.set_tensor(i['index'], Xte_in[k:k + 1]); it.invoke()
    errs.append(np.abs(it.get_tensor(o['index'])[0] - Pr[k].reshape(-1)).max() * IMG)
print('keras vs tflite 최대 차이: %.2f px' % max(errs))

In [ ]:
# 얼굴 검출기(YuNet) 내려받아 save/ 에 같이 두기
import urllib.request, shutil
DET = 'face_detection_yunet_2023mar.onnx'
if not os.path.exists(save_dir + DET):
    urllib.request.urlretrieve('https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx', save_dir + DET)
print(DET, '%.0f KB' % (os.path.getsize(save_dir + DET) / 1024), '(약 230KB 가 정상. 1KB 면 LFS 포인터 → opencv_zoo 에서 직접 받기)')
for f in ['wflw_mesh.tflite', DET, 'wflw_train_log.png', 'wflw_samples.png']:
    print('%-36s %s' % (f, 'OK' if os.path.exists(save_dir + f) else '없음'))

## 7. 다음 단계
`save/wflw_mesh.tflite` 와 `face_detection_yunet_2023mar.onnx` 를
1. 드라이브 `proj_face/FaceReading_3/` 에 복사 → **`Norms_Train.ipynb`** 실행 (우리 모델로 Face Shape Dataset 을 재서 norms.json·얼굴형 MLP 재생성)
2. 보드 `~/work/FaceReading_3/` 에 복사 → `python face_reading.py`
